# Sales Prediction Using Python
## Regression with Linear Regression and Random Forest

**Objective:** Predict sales using TV, Radio, and Newspaper advertising expenditure.

Dataset: `Advertising.csv` (200 observations). The notebook downloads a public copy automatically. You can also place your own `Advertising.csv` in the notebook folder.


## 1. Install and import libraries

In [ ]:
# If needed, uncomment this line:
# !pip install pandas numpy matplotlib seaborn scikit-learn

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42


## 2. Load the dataset

In [ ]:
# Use a local file if available; otherwise download a public copy.
DATA_URL = "https://raw.githubusercontent.com/selva86/datasets/master/Advertising.csv"

if os.path.exists("Advertising.csv"):
    df = pd.read_csv("Advertising.csv")
else:
    df = pd.read_csv(DATA_URL)

# Remove an optional index column often included in CSV versions.
df = df.loc[:, ~df.columns.astype(str).str.match(r"^Unnamed")]

# Standardize column names.
df.columns = [str(c).strip().capitalize() for c in df.columns]

print("Dataset shape:", df.shape)
display(df.head())


## 3. Data inspection and cleaning

In [ ]:
print("Data types:")
display(df.dtypes)

print("\\nMissing values:")
display(df.isnull().sum())

print("\\nDuplicate rows:", df.duplicated().sum())

print("\\nDescriptive statistics:")
display(df.describe())

# Convert required columns to numeric and remove invalid/missing records.
features = ["Tv", "Radio", "Newspaper"]
target = "Sales"

for col in features + [target]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=features + [target]).drop_duplicates().reset_index(drop=True)

print("Cleaned dataset shape:", df.shape)


## 4. Exploratory Data Analysis (EDA)

In [ ]:
# Target distribution
plt.figure(figsize=(7, 4))
sns.histplot(df["Sales"], kde=True)
plt.title("Distribution of Sales")
plt.xlabel("Sales")
plt.tight_layout()
plt.show()

# Pairplot of all features
sns.pairplot(df[features + [target]], diag_kind="hist")
plt.suptitle("Pairplot of Advertising Features and Sales", y=1.02)
plt.show()


### Individual scatter plots: Sales vs each advertising channel

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for ax, col in zip(axes, features):
    sns.scatterplot(data=df, x=col, y="Sales", ax=ax)
    sns.regplot(data=df, x=col, y="Sales", scatter=False, ax=ax, color="red")
    ax.set_title(f"Sales vs {col} Advertising")

plt.tight_layout()
plt.show()


### Correlation matrix heatmap

In [ ]:
corr = df[features + [target]].corr(numeric_only=True)

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

print("Correlation with Sales:")
display(corr["Sales"].drop("Sales").sort_values(ascending=False))


## 5. Prepare training and testing data

In [ ]:
X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


## 6. Train baseline model: Linear Regression

In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

linear_pred = linear_model.predict(X_test)

print("Intercept:", linear_model.intercept_)
print("Coefficients:")
display(pd.DataFrame({
    "Feature": features,
    "Coefficient": linear_model.coef_
}).sort_values("Coefficient", ascending=False))


## 7. Train additional model: Random Forest Regressor

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=300,
    random_state=RANDOM_STATE,
    min_samples_leaf=2
)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)


## 8. Evaluate both models: MAE, RMSE, and R²

In [ ]:
def evaluate_model(name, y_true, y_pred):
    return {
        "Model": name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred)
    }

results = pd.DataFrame([
    evaluate_model("Linear Regression", y_test, linear_pred),
    evaluate_model("Random Forest", y_test, rf_pred)
]).sort_values("RMSE")

display(results)

best_name = results.iloc[0]["Model"]
best_model = linear_model if best_name == "Linear Regression" else rf_model
best_pred = linear_pred if best_name == "Linear Regression" else rf_pred

print("Best model based on lowest test RMSE:", best_name)


## 9. Actual vs predicted sales

In [ ]:
plt.figure(figsize=(7, 6))
sns.scatterplot(x=y_test, y=best_pred)
lims = [
    min(y_test.min(), best_pred.min()),
    max(y_test.max(), best_pred.max())
]
plt.plot(lims, lims, "r--", label="Perfect prediction")
plt.xlabel("Actual Sales")
plt.ylabel("Predicted Sales")
plt.title(f"Actual vs Predicted Sales — {best_name}")
plt.legend()
plt.tight_layout()
plt.show()


## 10. Residual plot for the best model

In [ ]:
residuals = y_test - best_pred

plt.figure(figsize=(8, 5))
sns.scatterplot(x=best_pred, y=residuals)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Predicted Sales")
plt.ylabel("Residuals (Actual - Predicted)")
plt.title(f"Residual Plot — {best_name}")
plt.tight_layout()
plt.show()

print("Residual mean:", residuals.mean())


**Interpretation:** Residuals should be scattered randomly around zero with no clear curve, funnel, or other pattern. A visible pattern may indicate non-linearity, unequal error variance, or a model that is missing important information.

## 11. Which advertising channel has the highest impact?

In [ ]:
if best_name == "Linear Regression":
    # Coefficients show the expected change in sales for a one-unit
    # increase in a channel, holding the other channels constant.
    impact = pd.Series(linear_model.coef_, index=features)
    impact_name = "Linear regression coefficients"
else:
    # Impurity-based feature importance from the fitted forest.
    impact = pd.Series(rf_model.feature_importances_, index=features)
    impact_name = "Random Forest feature importance"

impact = impact.sort_values(ascending=False)
display(impact.rename("Importance / Coefficient").to_frame())

plt.figure(figsize=(7, 4))
sns.barplot(x=impact.values, y=impact.index)
plt.title(impact_name)
plt.xlabel("Value")
plt.ylabel("Advertising Channel")
plt.tight_layout()
plt.show()

print("Highest-ranked channel:", impact.index[0])


**Important:** Linear regression coefficients are directional and measured in sales units per advertising-spend unit. Random Forest feature importances are relative split-based scores, not causal effects. The result describes association in this dataset and does not prove that increasing a channel's budget will cause the same increase in sales.

## 12. Predict sales for a new advertising budget

In [ ]:
# Example values — replace with your own TV, Radio, and Newspaper spend.
new_campaign = pd.DataFrame({
    "Tv": [150.0],
    "Radio": [25.0],
    "Newspaper": [20.0]
})

predicted_sales = best_model.predict(new_campaign)[0]
print(f"Predicted sales: {predicted_sales:.2f}")


## 13. Final project conclusion

- The project predicts sales using TV, Radio, and Newspaper advertising expenditure.
- Linear Regression is used as the baseline model.
- Random Forest Regression is compared as a second model.
- MAE, RMSE, and R² are used to evaluate test-set performance.
- The best model is selected using the lowest test RMSE.
- The residual plot is checked for systematic error patterns.
- The channel with the highest coefficient or feature importance is reported, with the interpretation caveat above.

**Before submission:** Run every cell from top to bottom, inspect the actual metric values and plots, and write your conclusion using the results produced by your run. Do not copy assumed accuracy values.
